# Agent에 Payment Limit 활성화 — Strands

## 개요

이 튜토리얼에서는 **AgentCore payments SDK**와 **Strands Agents**를 사용하여 payment-enabled AI agent를 구축하는 방법을 보여줍니다. `AgentCorePaymentsPlugin`이 전체 x402 payment flow를 자동으로 처리하므로 developer가 payment logic을 작성할 필요가 없습니다.

### 내부 작동 방식

```
Agent (Strands + http_request tool)
  │
  ├─► http_request GET https://x402-test.genesisblock.ai/api/weather
  │                         │
  │                   Server가 HTTP 402 반환(x402 payment 필요)
  │                         │
  │         AgentCorePaymentsPlugin이 402 intercept
  │                         │
  │         ProcessPayment ─► budget 확인 ─► tx sign ─► proof 반환
  │                         │
  │         Plugin이 X-PAYMENT header로 http_request 재시도
  │                         │
  ├─► 200 OK ─ agent가 paid content 수신
  │
  └─► Agent가 사용자에게 결과 요약
```

developer 코드는 plugin을 생성하여 `http_request` tool과 함께 agent에 연결하고 호출합니다. agent는 HTTP를 통해 Coinbase Bazaar x402 endpoint를 직접 호출합니다. Tutorial 04에서는 MCP tool과 AgentCore Gateway를 통한 동일한 flow를 보여줍니다.

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)에서 무료 USDC를 받아 Base Sepolia(Ethereum)를 사용합니다. Testnet USDC는 현실 세계의 가치가 없습니다.

### Strands Agent — Payment Flow

![Strands Payment Flow](images/strands_payment_flow.png)


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                       |
|:--------------------|:----------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                          |
| Agent 유형          | Single                                                          |
| Agentic Framework   | Strands Agents                                                  |
| LLM 모델            | Anthropic Claude Sonnet                                         |
| 튜토리얼 구성 요소  | PaymentManager, PaymentsPlugin, Strands Agent, x402 endpoint    |
| 튜토리얼 분야       | 산업 공통                                                       |
| 예제 난이도         | 쉬움                                                            |
| 사용 SDK            | bedrock-agentcore SDK, Strands Agents SDK                       |

## 사전 요구 사항

* Tutorial 00 완료(`.env`에 manager ARN, connector, instrument, session 존재)
* https://faucet.circle.com/에서 wallet에 testnet USDC 입금 완료
* `pip install 'bedrock-agentcore[strands-agents]'`

이 Notebook은 Coinbase CDP 또는 Stripe(Privy) wallet provider 중 어느 것이든 사용할 수 있습니다. agent 코드는 같으며 Tutorial 00에서 설정한 `.env` 값만 다릅니다.


> **비용 안내:** 이 튜토리얼은 현실 세계의 가치가 없는 testnet USDC를 사용하지만 기본 AWS 리소스(AgentCore payments)에는 요금이 발생할 수 있습니다.

In [ ]:
%pip install -r requirements.txt --quiet

## AWS Credentials 검증

계속 진행하기 전에 AWS credentials가 구성되었는지 검증합니다. 다음과 같은 표준 방식을 사용할 수 있습니다.
- `aws configure` (access keys)
- `aws sso login --profile <your-profile>` (SSO)
- environment variable(`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`)
- IAM role(EC2/ECS에서 실행하는 경우)

In [ ]:
import os
import boto3

# 이름이 지정된 AWS profile을 사용하려면 다음 줄의 주석 해제
# os.environ['AWS_PROFILE'] = '<your-profile>'
AWS_REGION = "us-west-2"

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
print(f"✅ Authenticated as: {identity['Arn']}")
print(f"   Region: {session.region_name}")

## 1단계: .env에서 Config Load

Tutorial 00에서 resource ID를 `.env`에 기록했습니다. 표준 Python pattern인 `load_dotenv()`로 load합니다. 아래 agent 코드는 구성한 wallet provider와 관계없이 동일합니다.


In [ ]:
import sys

sys.path.append("..")
from dotenv import load_dotenv
from utils import load_tutorial_env, print_summary

load_dotenv(override=True)

config = load_tutorial_env()
PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

# single-provider 및 multi-provider config 모두 처리
if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
    CONNECTOR_ID = config["instruments"][PROVIDER]["connector_id"]
else:
    INSTRUMENT_ID = config["instrument_id"]
    CONNECTOR_ID = config.get("connector_id")
    PROVIDER = config.get("provider_type", "unknown")

NETWORK = os.environ.get("NETWORK", "ETHEREUM")

# plugin의 network preference를 위해 NETWORK를 CAIP-2 chain identifier에 mapping
# merchant가 여러 blockchain network를 지원할 때 선호할 network를 plugin에 지정
NETWORK_PREFS = (
    ["eip155:84532", "base-sepolia"] if NETWORK == "ETHEREUM" else ["solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1"]
)

print_summary(
    "Loaded from .env",
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    provider=PROVIDER,
    instrument_id=INSTRUMENT_ID,
)

## 2단계: Payment Session 및 Payment Plugin 생성

Session은 agent 지출의 budget과 time window를 정의합니다.

`AgentCorePaymentsPlugin`은 이 session을 사용하여 다음 작업을 수행합니다.

1. HTTP 402 payment requirement가 포함된 tool response **intercept**
2. AgentCore를 통해 `ProcessPayment`를 **호출**하여 transaction에 sign하고 proof 생성
3. payment proof를 첨부하여 원래 request **재시도**

이 모든 작업은 agent loop 내부에서 수행되며 LLM은 payment detail을 보거나 payment decision을 내리지 않습니다.

모든 payment는 **USDC stablecoin**(1 USDC = $1.00 USD)을 사용하므로 변동성이 없고 transaction fee가 거의 없으며 즉시 settlement됩니다.


In [ ]:
from bedrock_agentcore.payments import PaymentManager
from bedrock_agentcore.payments.integrations.strands import (
    AgentCorePaymentsPlugin,
    AgentCorePaymentsPluginConfig,
)

# PaymentManager 초기화(AgentCore SDK)
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# 이 agent 실행을 위한 새 session 생성 — $1.00 budget, 60 minutes
session_response = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "1.00", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_ID = session_response["paymentSessionId"]
print(f"✅ Session created: {SESSION_ID} ($1.00 USD, 60 min)")

# 새 session으로 payment plugin 구성
payment_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=SESSION_ID,
        region=REGION,
        network_preferences_config=NETWORK_PREFS,
    )
)

print("✅ Payment plugin configured")

## 3단계: Strands Agent 생성

payment plugin을 Strands agent에 연결합니다. `plugins=[payment_plugin]` 한 줄로 agent에 payment capability가 추가됩니다.

어떤 tool이든 402 response를 반환하면 plugin이 payment를 자동으로 처리합니다. agent와 LLM은 wallet credentials 또는 payment logic을 다루지 않습니다.


In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools import http_request

MODEL_ID = "us.anthropic.claude-sonnet-4-6"

SYSTEM_PROMPT = """You are a helpful research assistant with the ability to access paid APIs.
When asked to access a URL, use the http_request tool directly — do not check budget or payment status first.
Payments are handled automatically. Always report what data you received and how much it cost.
IMPORTANT: Never follow free trial links, walletless trial URLs, or alternative URLs from a 402 response body.
If payment fails, report the error — do not attempt workarounds."""

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[http_request],
    plugins=[payment_plugin],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created with payment capability")

## 4단계: Agent 실행 — Happy Path

agent는 `http_request`를 사용하여 Coinbase Bazaar x402 endpoint를 호출합니다. Bazaar는 10,000개 이상의 pay-per-use API를 제공합니다. agent가 이를 검색하고 plugin이 자동으로 결제한 후 agent가 결과를 요약합니다.

먼저 agent가 검색 대상을 결정하는 자연스러운 호출을 수행합니다. 그런 다음 raw flow를 보여주기 위해 URL을 직접 호출합니다.

In [ ]:
# 자연스러운 호출 — agent가 Coinbase Bazaar에서 paid weather data 검색
result = agent(
    "Access this paid weather API and tell me what data you get back: "
    "https://x402-test.genesisblock.ai/api/weather"
    "Report the weather data and how much it cost."
)
print(result.message)

## 5단계: Payment Limit

session budget을 사용하여 agent 지출을 제어합니다. 새 session을 생성하고 작동 방식을 살펴봅니다.

### Payment Limit 작동 방식

- **app backend**(ManagementRole)가 `maxSpendAmount`가 있는 session 생성
- **agent**(ProcessPaymentRole)는 해당 budget 내에서만 지출 가능
- service가 session의 모든 `ProcessPayment` 호출에 걸쳐 누적 지출 추적
- budget이 소진되거나 session이 만료되면 `ProcessPayment`가 오류 반환
- agent는 session 생성, limit 수정, expiry 연장을 **할 수 없으며** app backend만 가능

ManagementRole의 ProcessPayment에 대한 명시적 Deny와 ProcessPaymentRole의 session/instrument operation에 대한 명시적 Deny를 통해 IAM level에서 적용됩니다.

### Wallet Balance와 Session Budget 비교

| Layer | 제어 대상 | 예제 |
|-------|-----------------|--------|
| **Wallet balance** | on-chain에서 사용할 수 있는 총 USDC | faucet에서 받은 10 USDC |
| **Session budget** | 하나의 task에서 agent가 지출할 수 있는 최대 금액 | session당 $0.50 |

session budget이 항상 더 엄격한 제약입니다. wallet에 10 USDC가 있어도 session budget이 $0.50이면 agent는 $0.50만 지출할 수 있습니다. 남은 9.50 USDC는 이후 session을 위해 wallet에 유지됩니다. 이 방식으로 단일 task에서 전체 balance를 위험에 빠뜨리지 않고 agent가 자금이 있는 wallet에 액세스하도록 할 수 있습니다.

In [ ]:
from bedrock_agentcore.payments import PaymentManager

# App backend가 제한적인 budget으로 새 session 생성
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# $0.50 budget, 60-minute expiry로 session 생성
# SDK에서 사용하는 parameter: limits, expiry_time_in_minutes
# Raw API에서 사용하는 parameter: limits.maxSpendAmount, expiryTimeInMinutes
new_session = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.50", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
new_session_id = new_session["paymentSessionId"]
print(f"✅ New session created: {new_session_id}")
print("   Budget: $0.50 USD | Expiry: 60 minutes")

### 새 Session으로 Plugin 업데이트

새 session으로 새 plugin instance를 생성합니다. 실제로는 agent task마다 새 session을 생성합니다.

In [ ]:
budget_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=new_session_id,
        region=REGION,
        network_preferences_config=NETWORK_PREFS,
    )
)

budget_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[http_request],
    plugins=[budget_plugin],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent configured with $0.50 budget session")

### Agent 실행 및 지출 확인

In [ ]:
result = budget_agent(
    "Access this paid weather API and summarize the data: https://x402-test.genesisblock.ai/api/weather"
)
print(result.message)

In [ ]:
# $0.50 budget 중 사용한 금액 확인
session_info = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=new_session_id,
)

session = session_info
available = session.get("availableLimits", {}).get("availableSpendAmount", {})
print_summary(
    "Budget Status",
    session_id=new_session_id,
    remaining_budget=f"${available.get('value', 'N/A')} {available.get('currency', '')}",
    budget_limit=session.get("limits", {}).get("maxSpendAmount", "N/A"),
)

### Budget을 초과하면 어떻게 되나요?

매우 작은 budget($0.0001)으로 session을 생성하고 agent가 이보다 비싼 항목($0.001)을 결제하려 할 때 어떻게 되는지 확인합니다. service가 payment를 거부하며 budget은 infrastructure level에서 적용됩니다.


In [ ]:
# 매우 작은 budget으로 session 생성
tiny_session = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.0001", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
tiny_session_id = tiny_session["paymentSessionId"]
print(f"✅ Tiny session: {tiny_session_id} (budget: $0.0001 USD)")

In [ ]:
# 매우 작은 budget으로 agent 생성
tiny_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=tiny_session_id,
        region=REGION,
        network_preferences_config=NETWORK_PREFS,
    )
)

tiny_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[http_request],
    plugins=[tiny_plugin],
    system_prompt=SYSTEM_PROMPT,
)

# payment가 $0.0001 budget을 초과하므로 실패해야 함
try:
    result = tiny_agent("Access this paid weather API: https://x402-test.genesisblock.ai/api/weather")
    print(result.message)
except Exception as e:
    print("💰 Budget exceeded — payment rejected by the service:")
    print(f"   {e}")
    print("\n   This is the expected behavior. The budget is enforced at the infrastructure level.")
    print("   The budget is enforced by AgentCore payments, not by application code.")

agent가 결제를 시도했지만 payment 금액이 $0.0001 session budget을 초과하여 service가 거부했습니다. AgentCore payments가 infrastructure level에서 적용합니다.


### Payment Limit Pattern

| Pattern | Budget | Expiry | 사용 사례 |
|---------|--------|--------|----------|
| 빠른 조회 | $0.10 | 5 min | 단일 API 호출, 가격 확인 |
| research task | $1.00 | 60 min | multi-endpoint research session |
| 심층 분석 | $5.00 | 480 min | 확장된 multi-tool workflow |
| budget 상한 없음 | `limits` 생략 | 60 min | 신뢰할 수 있는 internal agent(주의해서 사용) |

agent가 수행할 작업에 적합한 budget으로 session을 생성합니다.

### Budget이 없는 Session(Spending Cap 없음)

`limits` field 없이 session을 생성할 수 있습니다. session은 여전히 `availableLimits.availableSpendAmount`를 통해 남은 budget을 추적하지만 상한을 적용하지 않습니다. hard limit 없이 audit trail이 필요한 신뢰할 수 있는 internal agent에 유용합니다.

In [ ]:
# budget 상한이 없는 session — 시간 범위 내에서 agent가 자유롭게 지출 가능
uncapped_session = manager.create_payment_session(
    user_id=USER_ID,
    expiry_time_in_minutes=60,
    # limit 없음 — 지출은 추적되지만 상한은 적용되지 않음
)
uncapped_id = uncapped_session["paymentSessionId"]
print(f"✅ Uncapped session: {uncapped_id}")
print("   No budget limit — spend tracked but not enforced")
print("   Expiry: 60 minutes")
print("\n   ⚠️  Use with caution — only for trusted internal agents")

### Payment Limit 내부 적용 방식

| Dimension | 작동 방식 |
|-----------|-------------|
| **누적 추적** | service가 호출별이 아니라 session의 모든 ProcessPayment 호출을 누적 합산 |
| **Currency 변환** | budget은 USD이고 payment는 USDC이며 적용 시 service가 변환 |
| **거부** | 누적 지출 + 다음 payment가 `maxSpendAmount`를 초과하면 ProcessPayment가 오류 반환 |
| **시간 만료** | `expiryTimeInMinutes` 이후에는 budget이 남아 있어도 ProcessPayment 실패 |
| **IAM 적용** | Agent(ProcessPaymentRole)는 session 생성, budget 수정, expiry 연장 불가. application level이 아닌 구조적 제약 |
| **사용자별 격리** | session은 `userId` 범위이며 서로 다른 사용자는 독립된 budget 사용 |
| **선택적 Budget** | 제한 없는 session에서는 `limits` 생략. 남은 budget은 `availableLimits.availableSpendAmount`로 추적 |

## 5b단계: Plugin Built-in Tool — Runtime에서 Payment Limit Query

`AgentCorePaymentsPlugin`은 agent가 Runtime에서 payment state를 query할 수 있는 built-in tool 세 개를 등록합니다.

| Tool | 기능 |
|------|-------------|
| `get_payment_session` | 남은 budget, expiry, 지출 확인 |
| `get_payment_instrument` | wallet detail(address, network, status) 가져오기 |
| `list_payment_instruments` | 사용자의 모든 instrument 목록 표시 |

agent는 이를 사용하여 정보에 근거한 결정을 내릴 수 있습니다. 예를 들어 비용이 높은 호출을 시도하기 전에 budget이 충분한지 확인할 수 있습니다.


In [ ]:
# built-in get_payment_session tool을 사용하여 자체 budget을 확인하도록 agent에 요청
result = budget_agent("How much budget do I have left in my current session?")
print(result.message)

agent가 남은 budget을 확인하기 위해 plugin에서 등록한 tool인 `get_payment_session`을 호출했습니다. plugin이 이 tool을 자동으로 제공하므로 추가 코드는 필요하지 않습니다.


In [ ]:
# built-in list_payment_instruments tool을 사용하여 사용 가능한 wallet 목록을 표시하도록 agent에 요청
result = budget_agent("What payment instruments (wallets) do I have available?")
print(result.message)

In [ ]:
# get_payment_instrument를 사용하여 현재 wallet detail을 가져오도록 agent에 요청
result = budget_agent(
    f"Get me the details of my payment instrument {INSTRUMENT_ID} — what network is it on and what is the wallet address?"
)
print(result.message)

세 tool 모두 plugin에서 자동으로 등록하므로 추가 코드는 필요하지 않습니다. agent는 budget 확인(`get_payment_session`), 사용 가능한 wallet 검색(`list_payment_instruments`), wallet detail 검사(`get_payment_instrument`)를 통해 Runtime에서 정보에 근거한 결정을 내릴 수 있습니다.


## 6단계: AgentCore Observability에서 Payment Trace 보기

agent의 모든 `ProcessPayment` 호출에서 trace가 생성됩니다. payment flow를 보려면 CloudWatch console을 엽니다.

1. [CloudWatch console](https://console.aws.amazon.com/cloudwatch/)을 엽니다.
2. 왼쪽 navigation에서 **X-Ray traces** > **Traces**를 선택합니다.
3. service name `bedrock-agentcore`로 filtering합니다.
4. trace를 선택하여 reserve budget, sign transaction, commit의 3단계 payment flow를 확인합니다.

**Logs** > **Log groups** > `/aws/vendedlogs/bedrock-agentcore/<your-payment-manager-id>`에서 vended log도 볼 수 있습니다.

각 log entry에는 API operation, user ID, session ID, transaction status가 표시됩니다.


In [ ]:
# Tutorial 00에서 아직 수행하지 않았다면 observability 활성화
# vended log와 trace를 활성화하려면 주석 해제

# import boto3
# from utils import enable_observability
# account_id = boto3.client('sts').get_caller_identity()['Account']
# obs = enable_observability(
#     resource_arn=PAYMENT_MANAGER_ARN,
#     resource_id=os.environ.get('PAYMENT_MANAGER_ID', PAYMENT_MANAGER_ARN.split('/')[-1]),
#     account_id=account_id,
#     region=REGION,
# )

# CloudWatch console에서 log 보기
PAYMENT_MANAGER_ID = os.environ.get("PAYMENT_MANAGER_ID", PAYMENT_MANAGER_ARN.split("/")[-1])
print(f"CloudWatch Logs: /aws/vendedlogs/bedrock-agentcore/{PAYMENT_MANAGER_ID}")
print(f"Console: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#logsV2:log-groups")
print(f"X-Ray:   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")

## Wallet 및 Network에 종속되지 않는 구조

위 agent 코드는 다음 항목에 따라 변경되지 않습니다.
- **wallet provider 종류** — Coinbase CDP 또는 Stripe Privy
- **blockchain network 종류** — Ethereum(Base Sepolia) 또는 Solana(Solana Devnet)

변경되는 것은 Tutorial 00에서 설정한 `.env`의 `INSTRUMENT_ID`와 `PAYMENT_CONNECTOR_ID`뿐입니다. 나머지는 plugin에서 처리합니다.

### 작동하는 이유

| Layer | Network 인식 여부 | 알고 있는 정보 |
|-------|---------------|---------------|
| PaymentManager | 아니요 | authorization policy만 인식 |
| PaymentConnector | 아니요 | chain이 아닌 provider(Coinbase/Privy) 종류 |
| PaymentInstrument | **예** | 생성 시 설정된 `network: ETHEREUM` 또는 `network: SOLANA` |
| ProcessPayment | **예** | merchant의 402 payload가 chain 지정(`eip155:84532` 또는 `solana:...`) |
| Agent code | **아니요** | instrument ID를 plugin에 전달 |

control plane(manager, connector, credentials)은 한 번 설정하면 모든 network에서 작동합니다. network를 전환하려면 data plane API를 한 번 호출하여 새 instrument를 생성하면 됩니다. control plane은 변경하지 않습니다.

### 선택 사항: Solana Instrument 생성(동일한 Manager, 다른 Network)

network에 종속되지 않음을 확인하려면 **동일한** manager와 connector를 사용하여 Solana에 두 번째 instrument를 생성합니다. control plane 변경은 필요하지 않습니다.


In [ ]:
# 선택 사항: 동일한 manager에 Solana instrument 생성
# 실행하려면 주석 해제 — faucet.circle.com의 Solana Devnet USDC 필요

# import boto3
# from utils import client_token
#
# dp_client = boto3.client('bedrock-agentcore',
#     region_name=REGION,
#     endpoint_url=os.environ.get('PAYMENTS_DP_ENDPOINT'))
#
# solana_instrument = dp_client.create_payment_instrument(
#     paymentManagerArn=PAYMENT_MANAGER_ARN,
#     paymentConnectorId=CONNECTOR_ID,
#     paymentInstrumentType='EMBEDDED_CRYPTO_WALLET',
#     paymentInstrumentDetails={'embeddedCryptoWallet': {
#         'network': 'SOLANA',  # ← 이 부분만 변경
#         'linkedAccounts': [{'email': {'emailAddress': os.environ.get('USER_EMAIL', 'user@example.com')}}],
#     }},
#     clientToken=client_token(),
# )
#
# solana_id = solana_instrument['paymentInstrument']['paymentInstrumentId']
# solana_addr = solana_instrument['paymentInstrument']['paymentInstrumentDetails']['embeddedCryptoWallet']['walletAddress']
# print(f'Solana instrument: {solana_id}')
# print(f'Solana wallet:     {solana_addr}')
# print(f'Fund at: https://faucet.circle.com/ → Solana Devnet → {solana_addr}')
#
# # 이 Instrument를 사용하려면 .env만 업데이트하세요.
# #   INSTRUMENT_ID=<solana_id>
# # 그런 다음 위의 agent cell을 다시 실행하세요. Agent 코드는 변경되지 않습니다.


### 핵심 요점

> **하나의 manager, 하나의 connector, 여러 network의 다수 instrument를 사용하며 agent 코드는 변경되지 않습니다.**

이는 AgentCore payments의 protocol 및 wallet에 종속되지 않는 설계입니다. `ProcessPayment` API는 instrument 뒤의 wallet 또는 network와 관계없이 x402 version 감지, transaction signing(Ethereum은 EIP-712, Solana는 native), budget enforcement를 추상화합니다.

직접 확인하려면 다음을 수행합니다.
1. **Coinbase**로 Tutorial 00 실행 → 이 Notebook 실행 → agent 결제 ✅
2. **Privy**로 Tutorial 00 실행 → 이 Notebook 재실행 → 동일한 agent 코드, 동일한 결과 ✅
3. **Solana** instrument 생성(위 셀) → `INSTRUMENT_ID` 교체 → 동일한 agent 코드, 다른 chain ✅


## 수행 결과

약 10 lines의 코드로 다음을 수행했습니다.

1. Tutorial 00에서 payment stack load
2. manager, instrument, session을 사용하여 `AgentCorePaymentsPlugin` 생성
3. `plugins=[payment_plugin]`으로 Strands agent에 연결
4. agent 실행 — content 비용을 자동 결제
5. budget 제한 session 생성 및 지출 추적 검증

agent에는 wallet credentials가 노출되지 않았고 LLM은 payment decision을 내리지 않았습니다. application code가 아니라 service가 budget을 적용했습니다.

### 다음 단계

* **Tutorial 02** — AgentCore CLI와 적절한 role 분리를 사용하여 이 agent를 AgentCore Runtime에 배포
* **Tutorial 03** — Wallet operation: delegation, funding, balance 확인, multi-session pattern
* **Tutorial 04** — AgentCore Gateway를 통해 Coinbase Bazaar의 paid MCP tool 검색 및 호출


## 리소스 정리

Session은 구성된 `expiryTimeInMinutes`가 지나면 자동으로 만료됩니다. 모든 payment resource(Manager, Connector, Instrument)를 삭제하려면 Tutorial 00의 cleanup 셀을 실행합니다.

# 축하합니다!